# Phase 1 Real Model Implementation Smoke: A2/A2b First Comparator Plan

Notebook này chuẩn bị bước **Phase 1 real model implementation smoke** cho comparator thực thi đầu tiên. Phạm vi cố ý hẹp: bắt đầu bằng **A2/A2b scalp-only** trước khi mở A2c/A2d/A3/A4.

Nguyên tắc liêm chính khoa học:

- Notebook này **không được tự tạo kết quả mô hình** nếu model runner chưa được implement trong repo.
- Mọi path/hash phải bám các source-of-truth đã khóa: Gate 2.5 prereg bundle, Phase 0.5 estimator final, Phase 1 readiness revision, Phase 1 smoke contract.
- A2/A2b chỉ dùng scalp EEG ở test time. Không được dùng iEEG/teacher/privileged targets trong baseline scalp-only.
- Đây là implementation smoke/debug, không phải full 15-fold substantive run và không được dùng để claim efficacy.

## Cơ sở từ tài liệu V5.5

Các ràng buộc được notebook này dùng làm cơ sở triển khai:

- Primary endpoint của Phase 1 là nested leave-one-subject-out theo subject, binary memory load 4 vs 8 trong maintenance period.
- Test-time chỉ được dùng scalp EEG; iEEG/teacher là privileged information ở train time và không được rò vào comparator scalp-only.
- A2 là scalp-only neural baseline.
- A2b là pooled scalp-only cross-subject baseline.
- EEGNet là backbone mainline cho A2/A2b/A2c/A3/A4 khi phù hợp, nhưng config hiện tại vẫn phải được kiểm tra hash/status trước khi chạy.
- Outer-test subject không được tham gia bất kỳ fit nào: preprocessing fit, normalization fit, ICA/PCA/alignment/calibration/model selection/threshold tuning.
- Headline privileged-value claim về sau cần comparator suite đầy đủ A2/A2b/A2c/A2d/A3/A4, calibration package, negative controls và influence report. A2/A2b smoke không đủ để claim.

In [ ]:
# ============================================================
# Cell 1: Mount Drive, clone/pull repo, run unit tests
# ============================================================
# Mục tiêu:
# - Mount Google Drive.
# - Đảm bảo repo private tồn tại tại /content/eeg-ds004752.
# - Pull code mới nhất.
# - Chạy unit tests trước khi đọc/ghi artifact.
#
# Lưu ý:
# - Nếu repo private chưa clone, cần GitHub token.
# - Token không được in ra output.
# ============================================================

from pathlib import Path
import base64
import getpass
import os
import subprocess

from google.colab import drive

drive.mount('/content/drive')

REPO_DIR = Path('/content/eeg-ds004752')
REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'


def run(cmd, cwd=None, check=True, capture_output=False, text=True):
    print('\n$', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=cwd, check=check, capture_output=capture_output, text=text)


def git_auth_header():
    token = os.environ.get('GITHUB_TOKEN')
    if not token:
        token = getpass.getpass('Nhập GitHub token cho repo private: ')
    basic = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    return f'Authorization: Basic {basic}'


if not REPO_DIR.exists():
    extra_header = git_auth_header()
    run(['git', '-c', f'http.extraHeader={extra_header}', 'clone', REPO_URL, str(REPO_DIR)])
else:
    print('Repo đã tồn tại:', REPO_DIR)

%cd /content/eeg-ds004752

pull_result = subprocess.run(['git', 'pull'], cwd=REPO_DIR)
if pull_result.returncode != 0:
    extra_header = git_auth_header()
    run(['git', '-c', f'http.extraHeader={extra_header}', 'pull'], cwd=REPO_DIR)

run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)
run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=REPO_DIR)

In [ ]:
# ============================================================
# Cell 2: Utility functions
# ============================================================
# Các hàm nhỏ để đọc JSON, tính SHA256, lấy Git commit và ghi artifact.
# Không có logic khoa học hoặc model training trong cell này.
# ============================================================

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import subprocess


def utc_timestamp():
    return datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')


def read_json(path: Path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Missing JSON file: {path}')
    return json.loads(path.read_text(encoding='utf-8'))


def write_json(path: Path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')


def sha256_file(path: Path) -> str:
    path = Path(path)
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def git_output(args):
    return subprocess.check_output(args, cwd=REPO_DIR, text=True).strip()


def git_status_record():
    status_short = git_output(['git', 'status', '--short'])
    return {
        'branch': git_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD']),
        'commit': git_output(['git', 'rev-parse', 'HEAD']),
        'working_tree_clean': status_short == '',
        'git_status_short': status_short,
    }

In [ ]:
# ============================================================
# Cell 3: Fixed source-of-truth paths
# ============================================================
# Mục tiêu:
# - Khóa các path đã được xác nhận ở các notebook trước.
# - Không tự động lấy latest.txt để tránh vô tình dùng run mới chưa được review.
# ============================================================

DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
DATASET_ROOT = DRIVE_ROOT / 'data' / 'ds004752'

EXPECTED_PREREG_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'

PREREG_RUN = DRIVE_ROOT / 'artifacts' / 'prereg' / '20260418T161442014597Z'
PREREG_BUNDLE = PREREG_RUN / 'prereg_bundle.json'
REAL_PHASE_SOT = PREREG_RUN / 'real_phase_source_of_truth.json'

PHASE05_ESTIMATOR_RUN = DRIVE_ROOT / 'artifacts' / 'phase05_estimators' / '20260419T130315366518Z'
PHASE1_READINESS_RUN = DRIVE_ROOT / 'artifacts' / 'phase1_readiness' / '20260419T154005857077Z'
PHASE1_SMOKE_RUN = DRIVE_ROOT / 'artifacts' / 'phase1_smoke' / '20260419T160919267039Z'
PHASE1_SMOKE_SOT = PHASE1_SMOKE_RUN / 'phase1_smoke_source_of_truth.json'

PHASE1_MODEL_PLAN_ROOT = DRIVE_ROOT / 'artifacts' / 'phase1_model_smoke_plan'

required_paths = {
    'dataset_root': DATASET_ROOT,
    'prereg_bundle': PREREG_BUNDLE,
    'real_phase_source_of_truth': REAL_PHASE_SOT,
    'phase05_estimator_run': PHASE05_ESTIMATOR_RUN,
    'phase1_readiness_run': PHASE1_READINESS_RUN,
    'phase1_smoke_run': PHASE1_SMOKE_RUN,
    'phase1_smoke_source_of_truth': PHASE1_SMOKE_SOT,
}

print('================ Required Paths ================')
for name, path in required_paths.items():
    exists = path.exists()
    print(name, path, 'exists =', exists)
    if not exists:
        raise FileNotFoundError(f'Missing required path: {name}: {path}')

In [ ]:
# ============================================================
# Cell 4: Verify prior source-of-truth chain
# ============================================================
# Mục tiêu:
# - Đảm bảo prereg bundle đúng LOCKED HASH đã ghi trong JSON/source-of-truth.
# - Ghi nhận raw file SHA256 riêng để audit, nhưng không dùng raw file hash làm prereg identity.
# - Đảm bảo Phase 1 readiness đã comparator-complete.
# - Đảm bảo Phase 1 smoke trước đó chỉ là smoke contract, không phải model result.
# - Đảm bảo Phase 0.5 estimator final đã có đủ 68/68 sessions.
#
# Vì sao không assert raw_file_sha256 == EXPECTED_PREREG_HASH?
# - prereg_bundle.json có thể chứa chính trường hash/metadata hoặc bị thay đổi byte-level
#   do ghi lại file, line ending, hoặc revision metadata.
# - Hash khóa khoa học phải là hash đã được bundle/source-of-truth ghi nhận, không phải
#   luôn luôn là SHA256 byte-level của file hiện tại.
# - raw_file_sha256 vẫn được in ra để truy vết liêm chính.
# ============================================================

prereg_bundle = read_json(PREREG_BUNDLE)
real_phase_sot = read_json(REAL_PHASE_SOT)
phase1_smoke_sot = read_json(PHASE1_SMOKE_SOT)

readiness_path = PHASE1_READINESS_RUN / 'phase1_input_freeze_revision.json'
if not readiness_path.exists():
    readiness_path = PHASE1_READINESS_RUN / 'phase1_input_freeze.json'
phase1_readiness = read_json(readiness_path)

phase05_summary = read_json(PHASE05_ESTIMATOR_RUN / 'phase05_estimators_summary.json')
phase05_feature_report = read_json(PHASE05_ESTIMATOR_RUN / 'feature_extraction_report.json')
phase05_exclusion_note = read_json(PHASE05_ESTIMATOR_RUN / 'phase05_estimator_exclusion_note.json')

raw_prereg_file_sha256 = sha256_file(PREREG_BUNDLE)
locked_prereg_hash = (
    prereg_bundle.get('prereg_bundle_hash_sha256')
    or prereg_bundle.get('bundle_hash_sha256')
    or prereg_bundle.get('locked_prereg_bundle_hash_sha256')
)
real_phase_prereg_hash = (
    real_phase_sot.get('prereg_bundle_hash_sha256')
    or real_phase_sot.get('locked_prereg_bundle_hash_sha256')
    or real_phase_sot.get('prereg_bundle', {}).get('sha256')
    or real_phase_sot.get('source_hashes_sha256', {}).get('prereg_bundle', {}).get('sha256')
)

hash_candidates = {
    'expected_prereg_hash': EXPECTED_PREREG_HASH,
    'locked_prereg_hash_from_bundle': locked_prereg_hash,
    'locked_prereg_hash_from_real_phase_sot': real_phase_prereg_hash,
    'raw_prereg_file_sha256': raw_prereg_file_sha256,
}

# Hard check: at least one locked source must match the expected prereg identity.
# raw_prereg_file_sha256 is informational and may differ.
if EXPECTED_PREREG_HASH not in {locked_prereg_hash, real_phase_prereg_hash}:
    raise AssertionError(
        'Locked prereg hash mismatch. Details:\n'
        + json.dumps(hash_candidates, indent=2, ensure_ascii=False)
    )

assert phase1_readiness.get('status') == 'phase1_input_freeze_revised_comparator_complete'
assert phase05_summary.get('n_sessions') == 68
assert phase05_exclusion_note.get('included_sessions') == 68
assert phase05_exclusion_note.get('excluded_sessions') == 0

# Phase 1 smoke source should explicitly be non-claim/non-model.
assert phase1_smoke_sot.get('decoder_trained') is False
assert phase1_smoke_sot.get('model_metrics_computed') is False
assert phase1_smoke_sot.get('claim_ready') is False

print('================ Source Chain Verified ================')
print('Expected prereg hash:', EXPECTED_PREREG_HASH)
print('Locked prereg hash from bundle:', locked_prereg_hash)
print('Locked prereg hash from real-phase SOT:', real_phase_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_file_sha256)
if raw_prereg_file_sha256 != EXPECTED_PREREG_HASH:
    print('NOTE: raw file SHA256 differs from locked prereg hash; this is recorded for audit and is not treated as prereg identity failure.')
print('Phase 1 readiness:', phase1_readiness.get('status'))
print('Phase 0.5 estimator status:', phase05_summary.get('status'))
print('Phase 0.5 sessions:', phase05_summary.get('n_sessions'))
print('Phase 1 smoke SOT:', PHASE1_SMOKE_SOT)
print('Decoder trained in prior smoke:', phase1_smoke_sot.get('decoder_trained'))

In [ ]:
# ============================================================
# Cell 5: Inspect comparator/config state for A2/A2b implementation
# ============================================================
# Mục tiêu:
# - Ghi hash các config liên quan.
# - Xác nhận rằng A2/A2b phải map vào EEGNet/scalp-only family theo comparator revision.
# - Không thay đổi config hoặc threshold trong notebook này.
# ============================================================

CONFIG_PATHS = {
    'eegnet': REPO_DIR / 'configs/models/eegnet.yaml',
    'scalp_minimal': REPO_DIR / 'configs/preprocess/scalp_minimal.yaml',
    'loso_subject': REPO_DIR / 'configs/split/loso_subject.yaml',
    'metrics': REPO_DIR / 'configs/eval/metrics.yaml',
    't4_safe': REPO_DIR / 'configs/runtime/t4_safe.yaml',
    'a100_fast': REPO_DIR / 'configs/runtime/a100_fast.yaml',
}

COMPARATOR_REVISION_REGISTRY = PREREG_RUN / 'phase1_comparator_revision' / 'phase1_comparator_revision_registry.json'
comparator_revision = read_json(COMPARATOR_REVISION_REGISTRY)

config_hashes = {}
print('================ Config Hashes ================')
for name, path in CONFIG_PATHS.items():
    if not path.exists():
        raise FileNotFoundError(f'Missing config: {name}: {path}')
    digest = sha256_file(path)
    config_hashes[name] = {'path': str(path), 'sha256': digest}
    print(name, digest, path)

print('\n================ Comparator Revision ================')
print('Registry:', COMPARATOR_REVISION_REGISTRY)
print('Status:', comparator_revision.get('status'))
print('Base prereg hash:', comparator_revision.get('base_prereg_bundle_hash_sha256'))
print('Available keys:', sorted(comparator_revision.keys()))
assert comparator_revision.get('status') == 'phase1_comparator_revision_locked'
assert comparator_revision.get('base_prereg_bundle_hash_sha256') == EXPECTED_PREREG_HASH

In [ ]:
# ============================================================
# Cell 6: Runtime/hardware preflight
# ============================================================
# Mục tiêu:
# - Ghi nhận GPU/CPU/RAM ở Colab.
# - Không bắt buộc A100 cho plan/preflight.
# - Khi model runner thật xuất hiện, A2/A2b 1-2 fold smoke có thể thử trên T4; full neural suite nên dùng A100.
# ============================================================

import platform
import shutil


# ----------------------------
# Repo freshness check
# ----------------------------
# Cell 1 đã clone/pull repo và chạy tests. Đoạn này pull lại nhẹ trước khi
# kiểm tra engine để tránh dùng code cũ nếu bạn chạy lại riêng các cell sau.
# Dùng --ff-only để không tạo merge commit trong Colab.

pull_result = subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)
if pull_result.returncode != 0:
    # Nếu repo private yêu cầu auth lại, dùng helper đã định nghĩa ở Cell 1.
    extra_header = git_auth_header()
    run(['git', '-c', f'http.extraHeader={extra_header}', 'pull', '--ff-only'], cwd=REPO_DIR)

print('Current Git commit:', git_output(['git', 'rev-parse', 'HEAD']))
print('Current Git branch:', git_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD']))


nvidia_smi = shutil.which('nvidia-smi')
gpu_report = None
if nvidia_smi:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
        text=True,
        capture_output=True,
        check=False,
    )
    gpu_report = result.stdout.strip() if result.returncode == 0 else result.stderr.strip()
else:
    gpu_report = 'nvidia-smi not found'

runtime_preflight = {
    'python': platform.python_version(),
    'platform': platform.platform(),
    'gpu_report': gpu_report,
    'recommendation': {
        'plan_or_blocker_cells': 'CPU/T4 is enough',
        'a2_a2b_one_or_two_fold_model_smoke': 'T4 may be acceptable if batch size is small; A100 preferred for faster iteration',
        'full_phase1_neural_suite': 'A100 recommended, especially for A2/A2b/A2c/A3/A4 and ablations',
    },
}

print('================ Runtime Preflight ================')
print(json.dumps(runtime_preflight, indent=2, ensure_ascii=False))

In [ ]:
# ============================================================
# Cell 7: Detect whether real A2/A2b model runner exists
# ============================================================
# Mục tiêu:
# - Kiểm tra repo đã có implementation runner thật cho Phase 1 model smoke chưa.
# - Nếu chưa có, notebook sẽ ghi blocker rõ ràng và không chạy training.
# ============================================================

candidate_runner_files = [
    REPO_DIR / 'src/phase1/model_smoke.py',
    REPO_DIR / 'src/phase1/model_runner.py',
    REPO_DIR / 'src/models/backbones/eegnet.py',
    REPO_DIR / 'src/models/comparators/a2.py',
    REPO_DIR / 'src/models/comparators/a2b.py',
]

cli_text = (REPO_DIR / 'src/cli.py').read_text(encoding='utf-8')
runner_file_status = {str(path.relative_to(REPO_DIR)): path.exists() for path in candidate_runner_files}
cli_support_status = {
    'phase1_real_command_present': 'phase1_real' in cli_text,
    'contract_smoke_flag_present': '--smoke' in cli_text,
    'model_smoke_flag_present': '--model-smoke' in cli_text or 'model_smoke' in cli_text,
    'comparator_arg_present': '--comparators' in cli_text,
}

model_runner_available = (
    runner_file_status.get('src/phase1/model_smoke.py', False)
    and cli_support_status['model_smoke_flag_present']
    and cli_support_status['comparator_arg_present']
)

engine_preflight = {
    'status': 'phase1_a2_a2b_model_runner_available' if model_runner_available else 'blocked_phase1_a2_a2b_model_runner_not_implemented',
    'runner_file_status': runner_file_status,
    'cli_support_status': cli_support_status,
    'required_before_training': [
        'CLI-backed runner for A2/A2b model smoke',
        'scalp EEG epoch loader with load 4 vs 8 maintenance labels',
        'subject-level LOSO fold builder with no outer-test fit leakage',
        'EEGNet or locked equivalent backbone implementation',
        'fold-level logits/metrics/calibration artifact writer',
    ],
}

print('================ Engine Preflight ================')
print(json.dumps(engine_preflight, indent=2, ensure_ascii=False))

## A2/A2b Implementation Smoke Plan

Comparator thực thi đầu tiên nên là **A2/A2b scalp-only**, vì đây là phần kiểm tra cơ sở trước khi mở privileged/distillation/CORAL:

- **A2**: scalp-only neural baseline, train/test theo subject-level LOSO, không dùng iEEG hoặc teacher.
- **A2b**: pooled scalp-only cross-subject baseline, vẫn giữ outer-test subject hoàn toàn ngoài fit/tuning/calibration.
- **Không chạy A3/A4** trong smoke đầu tiên vì A3/A4 cần teacher pathway và dễ tạo rủi ro leakage nếu baseline scalp-only chưa ổn.
- **Không chạy headline claim**. Smoke chỉ kiểm tra data loader, fold split, model loop, logging, calibration artifact shell, negative-control shell, và khả năng tái lập.

In [ ]:
# ============================================================
# Cell 8: Create A2/A2b model-smoke plan artifact
# ============================================================
# Mục tiêu:
# - Tạo một run folder plan/preflight có hash-link tới source-of-truth.
# - Ghi rõ comparator đầu tiên, scope smoke, artifact kỳ vọng và blocker nếu runner chưa có.
# ============================================================

run_dir = PHASE1_MODEL_PLAN_ROOT / utc_timestamp()
run_dir.mkdir(parents=True, exist_ok=False)

source_hashes = {
    'prereg_bundle': {'path': str(PREREG_BUNDLE), 'sha256': sha256_file(PREREG_BUNDLE)},
    'real_phase_source_of_truth': {'path': str(REAL_PHASE_SOT), 'sha256': sha256_file(REAL_PHASE_SOT)},
    'phase1_readiness': {'path': str(readiness_path), 'sha256': sha256_file(readiness_path)},
    'phase1_smoke_source_of_truth': {'path': str(PHASE1_SMOKE_SOT), 'sha256': sha256_file(PHASE1_SMOKE_SOT)},
    'phase05_estimators_summary': {
        'path': str(PHASE05_ESTIMATOR_RUN / 'phase05_estimators_summary.json'),
        'sha256': sha256_file(PHASE05_ESTIMATOR_RUN / 'phase05_estimators_summary.json'),
    },
    'phase05_estimator_exclusion_note': {
        'path': str(PHASE05_ESTIMATOR_RUN / 'phase05_estimator_exclusion_note.json'),
        'sha256': sha256_file(PHASE05_ESTIMATOR_RUN / 'phase05_estimator_exclusion_note.json'),
    },
    'phase1_comparator_revision_registry': {
        'path': str(COMPARATOR_REVISION_REGISTRY),
        'sha256': sha256_file(COMPARATOR_REVISION_REGISTRY),
    },
}

plan = {
    'status': 'phase1_a2_a2b_model_smoke_plan_recorded',
    'run_dir': str(run_dir),
    'created_utc': run_dir.name,
    'git': git_status_record(),
    'dataset_root': str(DATASET_ROOT),
    'source_hashes_sha256': source_hashes,
    'config_hashes_sha256': config_hashes,
    'runtime_preflight': runtime_preflight,
    'engine_preflight': engine_preflight,
    'first_comparators': [
        {
            'comparator_id': 'A2',
            'role': 'scalp_only_neural_baseline',
            'allowed_inputs': ['scalp_eeg_maintenance_epochs', 'memory_load_label_4_vs_8'],
            'forbidden_inputs': ['ieeg', 'beamforming_teacher_targets', 'outer_test_subject_fit_statistics'],
        },
        {
            'comparator_id': 'A2b',
            'role': 'pooled_scalp_only_cross_subject_baseline',
            'allowed_inputs': ['pooled_training_subject_scalp_eeg', 'memory_load_label_4_vs_8'],
            'forbidden_inputs': ['outer_test_subject_fit_statistics', 'ieeg', 'teacher_targets'],
        },
    ],
    'smoke_scope': {
        'outer_test_subjects': ['sub-01', 'sub-02'],
        'max_outer_folds': 2,
        'primary_denominator': 'subject',
        'task': 'binary_memory_load_4_vs_8',
        'window': 'maintenance_period',
        'non_claim_debug_only': True,
    },
    'minimum_future_training_artifacts': [
        'phase1_model_smoke_inputs.json',
        'fold_logs/fold_*.json',
        'a2_metrics_smoke.json',
        'a2b_metrics_smoke.json',
        'a2_a2b_logits_smoke/',
        'calibration_model_smoke_report.json',
        'negative_controls_model_smoke_report.json',
        'influence_model_smoke_report.json',
        'phase1_model_smoke_summary.json',
        'phase1_model_smoke_report.md',
    ],
    'scientific_limits': [
        'This plan does not train a decoder.',
        'This plan does not compute model performance.',
        'This plan does not estimate privileged-transfer efficacy.',
        'A2/A2b smoke cannot support a headline claim without the complete comparator suite and controls.',
    ],
}

plan_path = run_dir / 'phase1_a2_a2b_model_smoke_plan.json'
write_json(plan_path, plan)

report_lines = [
    '# Phase 1 A2/A2b Model-Smoke Plan',
    '',
    '## Status',
    '',
    f"- Status: `{plan['status']}`",
    f"- Engine preflight: `{engine_preflight['status']}`",
    f"- Run dir: `{run_dir}`",
    f"- Prereg bundle hash: `{EXPECTED_PREREG_HASH}`",
    '',
    '## First Comparators',
    '',
    '- A2: scalp-only neural baseline.',
    '- A2b: pooled scalp-only cross-subject baseline.',
    '',
    '## Smoke Scope',
    '',
    '- Outer folds: sub-01, sub-02 only.',
    '- Primary denominator remains subject.',
    '- Task: memory load 4 vs 8 during maintenance period.',
    '- No iEEG, teacher targets, or privileged information for A2/A2b.',
    '',
    '## Scientific Integrity',
    '',
    '- This file is a plan/preflight artifact, not a model result.',
    '- No decoder performance or privileged-transfer efficacy is claimed.',
    '- If the model runner is absent, training must remain blocked.',
]
report_path = run_dir / 'phase1_a2_a2b_model_smoke_plan.md'
report_path.write_text('\n'.join(report_lines) + '\n', encoding='utf-8')

latest_path = PHASE1_MODEL_PLAN_ROOT / 'latest.txt'
latest_path.write_text(str(run_dir) + '\n', encoding='utf-8')

print('================ A2/A2b Plan Recorded ================')
print('Run dir:', run_dir)
print('Plan:', plan_path)
print('Report:', report_path)
print('Engine status:', engine_preflight['status'])

In [ ]:
# ============================================================
# Cell 9: Controlled gate before any real model training
# ============================================================
# Mục tiêu:
# - Nếu runner chưa implement: ghi blocker và dừng an toàn.
# - Nếu runner đã implement sau này: người chạy vẫn phải bật flag thủ công.
#
# Không mặc định train model để tránh chạy ngoài ý muốn.
# ============================================================

RUN_A2_A2B_MODEL_SMOKE = False

blocker_path = run_dir / 'phase1_a2_a2b_model_smoke_blocker.json'
manual_hold_path = run_dir / 'phase1_a2_a2b_model_smoke_manual_hold.json'

if not model_runner_available:
    blocker = {
        'status': 'blocked_phase1_a2_a2b_model_runner_not_implemented',
        'run_dir': str(run_dir),
        'reason': 'Current repo has Phase 1 contract smoke only; no CLI-backed A2/A2b real model runner was detected.',
        'required_next_code_work': [
            'Implement src/phase1/model_smoke.py or equivalent runner.',
            'Expose CLI flags for --model-smoke and --comparators A2 A2b under phase1_real guard.',
            'Implement scalp EEG epoch loader for load 4 vs 8 maintenance trials.',
            'Implement leakage-safe LOSO fit/transform discipline.',
            'Write fold logs, logits, metrics, calibration, negative-control and influence artifact shells.',
            'Add unit tests before running Colab model smoke.',
        ],
        'not_ok_to_claim': [
            'No decoder was trained.',
            'No model metrics were computed.',
            'No privileged-transfer efficacy evidence exists from this notebook.',
        ],
    }
    write_json(blocker_path, blocker)
    print('================ Model Smoke Blocked ================')
    print('Status:', blocker['status'])
    print('Written:', blocker_path)
    print('NEXT: implement the CLI-backed A2/A2b model runner, then rerun this notebook.')
elif not RUN_A2_A2B_MODEL_SMOKE:
    manual_hold = {
        'status': 'manual_hold_before_a2_a2b_model_training',
        'run_dir': str(run_dir),
        'reason': 'Runner appears available, but RUN_A2_A2B_MODEL_SMOKE is False by default.',
        'required_manual_action': 'Review plan/source hashes, then set RUN_A2_A2B_MODEL_SMOKE = True to run 1-2 fold model smoke.',
    }
    write_json(manual_hold_path, manual_hold)
    print('================ Manual Hold ================')
    print('Status:', manual_hold['status'])
    print('Written:', manual_hold_path)
else:
    # This branch is intentionally a template. It should only be used after the repo implements
    # a real model-smoke CLI contract. Do not edit this command ad hoc after seeing results.
    output_root = DRIVE_ROOT / 'artifacts' / 'phase1_model_smoke'
    cmd = [
        'python', '-m', 'src.cli', 'phase1_real',
        '--profile', 't4_safe',
        '--config', str(PREREG_BUNDLE),
        '--readiness-run', str(PHASE1_READINESS_RUN),
        '--dataset-root', str(DATASET_ROOT),
        '--output-root', str(output_root),
        '--model-smoke',
        '--phase-config', 'configs/phase1/model_smoke.json',
        '--comparators', 'A2', 'A2b',
        '--max-trials-per-session', '24',
        '--max-outer-folds', '2',
        '--outer-test-subjects', 'sub-01', 'sub-02',
    ]
    run(cmd, cwd=REPO_DIR)
    print('A2/A2b model smoke command completed. Review generated artifacts before any further run.')

In [ ]:
# ============================================================
# Cell 10: Print closeout summary
# ============================================================
# Cell này chỉ diễn giải trạng thái hiện tại. Không thêm kết quả mô hình.
# ============================================================

print('================ Phase 1 A2/A2b Model-Smoke Closeout ================')
print('Plan source of truth:', run_dir)
print('Engine preflight:', engine_preflight['status'])
print('Model runner available:', model_runner_available)
print('Run flag:', RUN_A2_A2B_MODEL_SMOKE)
print('First comparators:', ['A2', 'A2b'])
print('Outer-test subjects planned:', ['sub-01', 'sub-02'])
print('Prereg hash:', EXPECTED_PREREG_HASH)

if not model_runner_available:
    print('\nBLOCKED: A2/A2b model runner is not implemented in the current repo.')
    print('NEXT: implement CLI-backed Phase 1 A2/A2b model-smoke runner and tests.')
elif not RUN_A2_A2B_MODEL_SMOKE:
    print('\nHELD: Runner appears available, but training was not launched because the manual flag is False.')
    print('NEXT: review the plan, then rerun with RUN_A2_A2B_MODEL_SMOKE = True if appropriate.')
else:
    print('\nCHECK REQUIRED: Model-smoke command was launched. Review fold logs and metrics before continuing.')

print('\nNOT OK TO CLAIM: This notebook does not by itself prove decoder efficacy or privileged-transfer efficacy.')